In [2]:
# search_hnsw.py
import hnswlib
import pickle
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
from feature_extraction import extract_features # Your existing function

# --- Configuration ---
index_file = 'sweater_hnsw_ResNet50_Yolo.bin'
pattern_ids_file = 'pattern_ids_yolo_seg.pkl'
feature_dim = 1024 # Should match the dimension used for building the index

def recommend_similar_images(query_image_path, hnsw_index, pattern_ids, num_recommendations=5):
    """
    Finds and displays the most similar images to a query image using an HNSWlib index.
    """
    print(f"--- Generating recommendations for {os.path.basename(query_image_path)} ---")
    
    # --- 1. Extract features from the query image ---
    _, query_vector = extract_features(query_image_path)
    if query_vector is None or not isinstance(query_vector, np.ndarray):
        print(f"Could not extract features from {query_image_path}")
        return

    # HNSWlib requires a 1D or 2D array, and it must be float32
    query_vector = np.array(query_vector).astype('float32')
    
    # --- 2. Search the HNSWlib index ---
    # The knn_query method returns labels and distances for the k nearest neighbors
    labels, distances = hnsw_index.knn_query(query_vector, k=num_recommendations)

    # The result is a 2D array, so we get the first (and only) row
    neighbor_indices = labels[0]
    
    print(f"Found neighbors with indices: {neighbor_indices}")
    
    # --- 3. Display the results ---
    plt.figure(figsize=(15, 5))

    # Display Query Image
    ax = plt.subplot(1, num_recommendations + 1, 1)
    query_img = Image.open(query_image_path)
    ax.imshow(query_img)
    ax.set_title("Query Image")
    ax.axis("off")

    # Display Recommended Images
    for i, idx in enumerate(neighbor_indices):
        recommended_pattern_id = pattern_ids[idx]
        
        # IMPORTANT: Update this path to how your images are stored.
        # This example assumes a structure like: data_directory/pattern_id/image.jpg
        rec_img_folder = os.path.join('/Volumes/Extreme Pro/ANN_photos', recommended_pattern_id)
        
        # Find the first image in the recommended pattern folder to display
        try:
            image_files = [f for f in os.listdir(rec_img_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if not image_files:
                print(f"Warning: No images found for pattern {recommended_pattern_id}.")
                continue
            first_image_file = image_files[0]
            rec_img_path = os.path.join(rec_img_folder, first_image_file)

            ax = plt.subplot(1, num_recommendations + 1, i + 2)
            rec_img = Image.open(rec_img_path)
            ax.imshow(rec_img)
            ax.set_title(f'Rec #{i+1}\n{recommended_pattern_id}')
            ax.axis('off')
        except (IOError, IndexError) as e:
            print(f"Warning: Could not load image for pattern {recommended_pattern_id}. Error: {e}")


    plt.tight_layout()
    plt.show()



In [3]:
import os
import pickle
import hnswlib
import numpy as np
import time

# --- Configuration ---
test_image_folder = 'Evaluation_Sweaters/pullovers'
valid_image_extensions = ('.jpg', '.jpeg', '.png')

num_recommendations = 10


# --- Load HNSWlib index and pattern IDs ---
try:
    print(f"Loading pattern IDs from {pattern_ids_file}...")
    with open(pattern_ids_file, 'rb') as f:
        pattern_ids = pickle.load(f)

    print(f"Loading HNSWlib index from {index_file}...")
    num_elements = len(pattern_ids)
    
    hnsw_index = hnswlib.Index(space='cosine', dim=feature_dim)
    hnsw_index.load_index(index_file, max_elements=num_elements)
    
    hnsw_index.set_ef(1000) 
    print(f"Index loaded successfully. Set efSearch = 1000")
        
except (IOError, RuntimeError) as e:
    print(f"Error loading files: {e}")
    print("Please ensure you have run your index-building script first.")
    exit()

# --- Lists to store results for averaging ---
all_query_times_ms = []
all_hits = [] # 1 for a hit, 0 for a miss

# --- Run Recommendation for all images in the folder ---
if not os.path.isdir(test_image_folder):
    print(f"Error: Test image folder not found at '{test_image_folder}'")
else:
    print(f"\n--- Starting batch recommendation for all images in {test_image_folder} ---")
    
    # Get a clean list of just the image files
    image_files_to_process = [
        f for f in sorted(os.listdir(test_image_folder)) 
        if f.lower().endswith(valid_image_extensions)
    ]
    total_images = len(image_files_to_process)
    
    if total_images == 0:
        print(f"No valid images found in {test_image_folder}")
        exit()

    for i, image_filename in enumerate(image_files_to_process):
        
        test_image_path = os.path.join(test_image_folder, image_filename)
        print(f"\n=============================================")
        print(f"Testing Image {i+1}/{total_images}: {image_filename}")
        print(f"=============================================")
        
        try:
            # --- Step 1: Get the correct pattern ID from the filename ---
            
            # Get the ID (e.g., "123456") from the filename (e.g., "123456.jpg")
            id_from_file = image_filename.split('.')[0] # <-- CHANGED
            
            # Format it to match the IDs in your pattern_ids list (e.g., "pattern_123456")
            correct_pattern_id = f"pattern_{id_from_file}" # <-- CHANGED
            
            print(f"  > Ground Truth ID: {correct_pattern_id}")

            # --- Step 2: Extract features for the query image ---
            img_path_out, query_vector = extract_features(test_image_path)
            
            if not isinstance(query_vector, np.ndarray):
                print(f"  > ERROR: Failed to extract features. Skipping.")
                print(f"  > Details: {query_vector}")
                continue 

            # --- Step 3: Run the query and time it ---
            start_time = time.perf_counter()
            labels, distances = hnsw_index.knn_query(query_vector, k=num_recommendations)
            end_time = time.perf_counter()
            
            query_time_ms = (end_time - start_time) * 1000 
            all_query_times_ms.append(query_time_ms)
            print(f"  > Query Time: {query_time_ms:.4f} ms")

            # --- Step 4: Check for a "hit" ---
            recommended_pattern_ids = [pattern_ids[i] for i in labels[0]]
            print(f"  > Recommendations: {recommended_pattern_ids}")
            
            # This check will now compare "pattern_123456" with the list
            if correct_pattern_id in recommended_pattern_ids:
                all_hits.append(1)
                print(f"  > RESULT: HIT! Correct ID '{correct_pattern_id}' was found.")
            else:
                all_hits.append(0)
                print(f"  > RESULT: MISS. Correct ID '{correct_pattern_id}' not found.")
                
        except Exception as e:
            print(f"ERROR: Could not process {image_filename}: {e}")

    # --- End of loop ---

    print("\n--- Batch recommendation complete. ---")

    # --- Calculate and print final averages ---
    if not all_hits:
        print("No images were processed successfully.")
    else:
        total_processed = len(all_hits)
        num_hits = np.sum(all_hits)
        avg_accuracy = (num_hits / total_processed) * 100
        avg_time = np.mean(all_query_times_ms)
        
        print("\n================== FINAL RESULTS ==================")
        print(f"Total Images Processed: {total_processed}")
        print(f"Total Hits (Recall@{num_recommendations}): {num_hits}")
        print(f"Average Accuracy (R@{num_recommendations}): {avg_accuracy:.2f}%")
        print(f"Average Query Time: {avg_time:.4f} ms")
        print("=================================================")

Loading pattern IDs from pattern_ids_yolo_seg.pkl...
Loading HNSWlib index from sweater_hnsw_ResNet50_Yolo.bin...
Index loaded successfully. Set efSearch = 1000

--- Starting batch recommendation for all images in Evaluation_Sweaters/pullovers ---

Testing Image 1/50: 1001132.jpeg
  > Ground Truth ID: pattern_1001132
  -> Final image shape: (336, 379, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 624ms/step
  > Query Time: 5.2090 ms
  > Recommendations: ['pattern_112498', 'pattern_5444', 'pattern_11562', 'pattern_7365384', 'pattern_439806', 'pattern_1021796', 'pattern_1080171', 'pattern_531287', 'pattern_803926', 'pattern_564348']
  > RESULT: MISS. Correct ID 'pattern_1001132' not found.

Testing Image 2/50: 100437.jpeg
  > Ground Truth ID: pattern_100437
  -> Final image shape: (434, 463, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
  > Query Time: 4.4808 ms
  > Recommendations: ['pattern_905868', 'pattern_772597', 'pattern_780598', 'pattern_1059845', 'pattern_100437', 'pattern_905464', 'pattern_800305'

In [4]:
import os
import pickle
import hnswlib
import numpy as np
import time

# --- Configuration ---
test_image_folder = 'Evaluation_Sweaters/cardigans'
valid_image_extensions = ('.jpg', '.jpeg', '.png')

num_recommendations = 10


# --- Load HNSWlib index and pattern IDs ---
try:
    print(f"Loading pattern IDs from {pattern_ids_file}...")
    with open(pattern_ids_file, 'rb') as f:
        pattern_ids = pickle.load(f)

    print(f"Loading HNSWlib index from {index_file}...")
    num_elements = len(pattern_ids)
    
    hnsw_index = hnswlib.Index(space='cosine', dim=feature_dim)
    hnsw_index.load_index(index_file, max_elements=num_elements)
    
    hnsw_index.set_ef(1000) 
    print(f"Index loaded successfully. Set efSearch = 1000")
        
except (IOError, RuntimeError) as e:
    print(f"Error loading files: {e}")
    print("Please ensure you have run your index-building script first.")
    exit()

# --- Lists to store results for averaging ---
all_query_times_ms = []
all_hits = [] # 1 for a hit, 0 for a miss

# --- Run Recommendation for all images in the folder ---
if not os.path.isdir(test_image_folder):
    print(f"Error: Test image folder not found at '{test_image_folder}'")
else:
    print(f"\n--- Starting batch recommendation for all images in {test_image_folder} ---")
    
    # Get a clean list of just the image files
    image_files_to_process = [
        f for f in sorted(os.listdir(test_image_folder)) 
        if f.lower().endswith(valid_image_extensions)
    ]
    total_images = len(image_files_to_process)
    
    if total_images == 0:
        print(f"No valid images found in {test_image_folder}")
        exit()

    for i, image_filename in enumerate(image_files_to_process):
        
        test_image_path = os.path.join(test_image_folder, image_filename)
        print(f"\n=============================================")
        print(f"Testing Image {i+1}/{total_images}: {image_filename}")
        print(f"=============================================")
        
        try:
            # --- Step 1: Get the correct pattern ID from the filename ---
            
            # Get the ID (e.g., "123456") from the filename (e.g., "123456.jpg")
            id_from_file = image_filename.split('.')[0] # <-- CHANGED
            
            # Format it to match the IDs in your pattern_ids list (e.g., "pattern_123456")
            correct_pattern_id = f"pattern_{id_from_file}" # <-- CHANGED
            
            print(f"  > Ground Truth ID: {correct_pattern_id}")

            # --- Step 2: Extract features for the query image ---
            img_path_out, query_vector = extract_features(test_image_path)
            
            if not isinstance(query_vector, np.ndarray):
                print(f"  > ERROR: Failed to extract features. Skipping.")
                print(f"  > Details: {query_vector}")
                continue 

            # --- Step 3: Run the query and time it ---
            start_time = time.perf_counter()
            labels, distances = hnsw_index.knn_query(query_vector, k=num_recommendations)
            end_time = time.perf_counter()
            
            query_time_ms = (end_time - start_time) * 1000 
            all_query_times_ms.append(query_time_ms)
            print(f"  > Query Time: {query_time_ms:.4f} ms")

            # --- Step 4: Check for a "hit" ---
            recommended_pattern_ids = [pattern_ids[i] for i in labels[0]]
            print(f"  > Recommendations: {recommended_pattern_ids}")
            
            # This check will now compare "pattern_123456" with the list
            if correct_pattern_id in recommended_pattern_ids:
                all_hits.append(1)
                print(f"  > RESULT: HIT! Correct ID '{correct_pattern_id}' was found.")
            else:
                all_hits.append(0)
                print(f"  > RESULT: MISS. Correct ID '{correct_pattern_id}' not found.")
                
        except Exception as e:
            print(f"ERROR: Could not process {image_filename}: {e}")

    # --- End of loop ---

    print("\n--- Batch recommendation complete. ---")

    # --- Calculate and print final averages ---
    if not all_hits:
        print("No images were processed successfully.")
    else:
        total_processed = len(all_hits)
        num_hits = np.sum(all_hits)
        avg_accuracy = (num_hits / total_processed) * 100
        avg_time = np.mean(all_query_times_ms)
        
        print("\n================== FINAL RESULTS ==================")
        print(f"Total Images Processed: {total_processed}")
        print(f"Total Hits (Recall@{num_recommendations}): {num_hits}")
        print(f"Average Accuracy (R@{num_recommendations}): {avg_accuracy:.2f}%")
        print(f"Average Query Time: {avg_time:.4f} ms")
        print("=================================================")

Loading pattern IDs from pattern_ids_yolo_seg.pkl...
Loading HNSWlib index from sweater_hnsw_ResNet50_Yolo.bin...
Index loaded successfully. Set efSearch = 1000

--- Starting batch recommendation for all images in Evaluation_Sweaters/cardigans ---

Testing Image 1/50: 100136.jpeg
  > Ground Truth ID: pattern_100136
  -> Final image shape: (420, 412, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
  > Query Time: 5.0213 ms
  > Recommendations: ['pattern_90027', 'pattern_171', 'pattern_8856', 'pattern_399437', 'pattern_7317989', 'pattern_1225168', 'pattern_7843', 'pattern_1338199', 'pattern_1084877', 'pattern_305779']
  > RESULT: MISS. Correct ID 'pattern_100136' not found.

Testing Image 2/50: 1028254.jpeg
  > Ground Truth ID: pattern_1028254
  -> No masks or boxes found. Falling back to original image.
  -> Final image shape: (480, 640, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
  > Query Time: 4.8968 ms
  > Recommendations: ['pattern_3377', 'pattern_671840', 'pattern_554684', 'pattern_10888', 